# Notebook 3 — The Loan Agent

**Companion notebook to:** *From Exam Question to Autonomous Agent: Teaching Agentic AI Through Financial
Mathematics* (CAS Global Teaching Materials Innovation Challenge submission)

This notebook builds and runs the agent described in **Section 4.3** of the case study: an agent that computes a
level loan payment and a full amortization schedule, including the drop/balloon final payment caused by rounding.

**Before running:** get a free Gemini API key at https://aistudio.google.com/app/apikey and paste it into the
`GEMINI_API_KEY` cell below (or set it as a Colab secret named `GEMINI_API_KEY`).

## 0. Setup

This notebook runs unchanged in **Google Colab** or in a **local editor** (VS Code, PyCharm, JupyterLab) using the
`uv`-managed environment that ships alongside these notebooks (`pyproject.toml` + `uv.lock`). The cell below
detects which one it is running in and does the right thing automatically:

- **Colab**: installs the required packages directly into the Colab runtime (nothing to download beforehand).
- **Local**: assumes you already ran `uv sync` once in the project folder (see `README.md`), so the packages are
  already present in `.venv` — this cell skips installation and just confirms the imports work.

Either way, run this cell once per session.

In [ ]:
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "agno[google,opentelemetry,sqlite]", "google-genai",
            "openinference-instrumentation-agno", "python-dotenv",
        ],
        check=True,
    )
else:
    print(
        "Running outside Colab -- assuming packages were already installed via "
        "`uv sync` in this project's folder (see README.md). Skipping pip install."
    )

import agno, google.genai, dotenv  # noqa: F401 -- import check only
print("Environment ready.")

In [ ]:
import os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Option A (recommended): click the key icon in Colab's left sidebar, add a
    # secret named GEMINI_API_KEY, and this line picks it up automatically.
    try:
        from google.colab import userdata
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass
    # Option B: no Colab secret found -- paste your key directly here instead.
    if not os.environ.get("GEMINI_API_KEY"):
        os.environ["GEMINI_API_KEY"] = "PASTE_YOUR_GEMINI_API_KEY_HERE"
else:
    # Local: reads GEMINI_API_KEY from a ".env" file in the project root.
    # Copy .env.example to .env and fill it in once -- see README.md.
    from dotenv import load_dotenv
    load_dotenv()

assert os.environ.get("GEMINI_API_KEY") and "PASTE_YOUR" not in os.environ["GEMINI_API_KEY"], (
    "GEMINI_API_KEY is not set. In Colab: add a secret named GEMINI_API_KEY, or "
    "paste your key directly into this cell. Locally: copy .env.example to .env "
    "in the project folder and add your key there."
)
print("GEMINI_API_KEY is set.")

## 1. The Financial Mathematics and the math (Section 4.3)

A loan is an annuity viewed from the borrower's side: the principal `L` is the present value of the promised
payments. For a level payment `PMT` over `n` periods at rate `i` per period:

$$L = PMT \times a_{\overline{n}|}, \qquad PMT = \frac{L}{a_{\overline{n}|}}$$

Amortization tracks the loan's balance period by period. Each payment covers interest on the balance since the
last payment, with the remainder reducing principal:

$$\text{Interest}_t = i \times B_{t-1}, \qquad \text{Principal}_t = PMT - \text{Interest}_t, \qquad B_t = B_{t-1} - \text{Principal}_t$$

Rounding a computed payment to a convenient figure means a level payment applied every period will not, in
general, bring the balance to exactly zero — a final drop or balloon payment absorbs the residual.

## 2. Python implementation (Section 4.3)

In [ ]:
from agno.exceptions import RetryAgentRun


def annuity_immediate_pv(payment: float, rate_per_period: float, n: int) -> float:
    """
    Compute the present value of a level annuity-immediate.

    Args:
        payment (float): The level payment amount per period. Must be
            non-negative.
        rate_per_period (float): The effective interest rate per
            payment period, as a decimal. Must be greater than -1.
        n (int): The number of payment periods. Must be positive.

    Returns:
        float: The present value of the annuity.
    """
    if payment < 0:
        raise RetryAgentRun("payment must be non-negative. Re-check the request.")
    if n <= 0:
        raise RetryAgentRun("n must be a positive number of periods. Re-check the request.")
    if rate_per_period <= -1:
        raise RetryAgentRun("rate_per_period must be greater than -100%. Re-check the request.")
    if rate_per_period == 0:
        return payment * n
    v = 1 / (1 + rate_per_period)
    a_n = (1 - v ** n) / rate_per_period
    return payment * a_n


def loan_payment(principal: float, rate_per_period: float, n: int) -> float:
    """
    Compute the level payment that amortises a loan over n periods.

    Args:
        principal (float): The loan amount. Must be positive.
        rate_per_period (float): The effective interest rate per
            payment period, as a decimal. Must be greater than -1.
        n (int): The number of payment periods. Must be positive.

    Returns:
        float: The level payment amount.
    """
    a_n = annuity_immediate_pv(1.0, rate_per_period, n)
    return principal / a_n


def amortisation_schedule(
    principal: float, rate_per_period: float, n: int, payment_amount: float
) -> list:
    """
    Build a complete amortisation schedule for a loan, correctly
    handling a final drop or balloon payment caused by rounding.

    Args:
        principal (float): The loan amount. Must be positive.
        rate_per_period (float): The effective interest rate per
            payment period, as a decimal. Must be greater than -1.
        n (int): The number of payment periods. Must be positive.
        payment_amount (float): The level payment used for every
            period except possibly the last. Must be positive.

    Returns:
        list: A list of n dictionaries, each with keys "period",
            "payment", "interest", "principal", and "balance".
    """
    schedule = []
    balance = round(principal, 2)
    for t in range(1, n + 1):
        interest = round(balance * rate_per_period, 2)
        if t < n:
            payment = payment_amount
            principal_paid = round(payment - interest, 2)
        else:
            # Final payment: whatever value exactly zeroes the
            # balance, absorbing any rounding residual.
            payment = round(balance + interest, 2)
            principal_paid = balance
        balance = round(balance - principal_paid, 2)
        schedule.append({
            "period": t, "payment": payment, "interest": interest,
            "principal": principal_paid, "balance": balance,
        })
    return schedule


# Sanity check against the case study's own worked example (Section 4.3):
# a $10,000 loan over 5 years at 5%, payment rounded up to $2,310.
for row in amortisation_schedule(10000, 0.05, 5, 2310):
    print(row)

Because rounding the exact payment up to $2,310 slightly overpays the loan at the level rate, the final payment
drops below the level amount — by $1.39, exactly enough to zero the balance. Compare the printed rows above to
the case study's Section 4.3 table.

## 3. Agentic integration (Section 4.3)

In [ ]:
from agno.agent import Agent
from agno.models.google import Gemini

loan_agent = Agent(
    name="Loan Agent",
    role="Answers loan questions: level payment amount, outstanding "
         "balance, interest/principal decomposition, full "
         "amortisation schedules, and implied interest rate.",
    model=Gemini(id="gemini-3.5-flash", temperature=0.0),
    tools=[loan_payment, amortisation_schedule],
    instructions=[
        "Always use one of the available tools to perform any "
        "numerical calculation. Never state a computed numeric "
        "result unless it came directly from a tool call.",
        "Every rate passed to a tool must be the effective rate per "
        "payment period. If a loan is quoted at an annual rate with "
        "monthly payments, convert to a monthly rate first and "
        "state the convention you used.",
    ],
    markdown=True,
)

loan_agent.print_response(
    "I'm taking out a $10,000 loan at 5% effective annual interest, "
    "repaid over 5 years with level annual payments rounded to the "
    "nearest dollar. Show me the full amortisation schedule."
)

The expected trajectory: the agent calls `loan_payment(10000, 0.05, 5)`, rounds the result to $2,310, then calls
`amortisation_schedule(10000, 0.05, 5, 2310)` and reports the five-row table, including the drop payment on the
final row. This is also the case study's example of a tool returning a *table* rather than a single number — every
row the agent reads back into its own context has a token cost (Section 3.8), which is why a production tool
might return a summary by default with a separate tool for the full detail.

## 4. Reliability evaluation (Section 3.7)

In [ ]:
from agno.eval.reliability import ReliabilityEval

response = loan_agent.run(
    "I'm taking out a $10,000 loan at 5% effective annual interest, "
    "repaid over 5 years with level annual payments rounded to the "
    "nearest dollar. Show me the full amortisation schedule."
)

ReliabilityEval(
    name="Loan Agent: payment then schedule, in order",
    agent_response=response,
    expected_tool_calls=["loan_payment", "amortisation_schedule"],
).run(print_results=True).assert_passed()

## Next

Continue with **Notebook 4 (Multi-Agent Team, Section 5)**, which coordinates specialists across bond pricing and
duration — a two-tool-family question that a single agent could still handle, but which starts to show why a
larger toolkit gets split across specialists.